In [8]:
# 🎬 🎓 SCENARIO: “College Smart Assistant System”
# 🏫 Background Story

# A college builds an AI-powered student assistant.

# 👉 Students can ask:

# “What is my attendance?”
# “What are my marks?”

# 👉 Instead of manually checking portals,
# 👉 AI fetches it instantly.

import gradio as gr

# 1. Database
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88},
    "103": {"name":"Ayan","attendance":100,"marks":100}
}

# 2. Tools
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"

def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"

def get_name(student_id):
    if student_id in students:
        return f" name: {students[student_id]['name']}"

# 3. Logic
def mcp_agent(message, student_id, history):
    if history is None:
        history = []

    # NEW FORMAT: Dictionary instead of Tuple
    history.append({"role": "user", "content": message})

    if not student_id:
        response = "⚠️ Please enter a Student ID first."
    else:
        msg_clean = message.lower()
        if "attendance" in msg_clean:
            response = get_attendance(student_id)
        elif "marks" in msg_clean:
            response = get_marks(student_id)
        elif "name" in msg_clean:
            response = get_name(student_id)
        else:
            response = "🤖 Try asking for 'attendance' or 'marks'."

    # NEW FORMAT: Dictionary instead of Tuple
    history.append({"role": "assistant", "content": response})

    return "", history

# 4. UI
with gr.Blocks() as demo:
    gr.Markdown("# 🎓 Student Assistant")
    
    sid = gr.Textbox(label="Student ID")
    # Note: No 'type' argument here to avoid the TypeError
    chatbot = gr.Chatbot() 
    msg = gr.Textbox(label="Your Question")

    msg.submit(mcp_agent, [msg, sid, chatbot], [msg, chatbot])

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


### With LLM

In [13]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup Groq Client
load_dotenv()
# Securely pulling the key you set in your environment earlier
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") 

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# 2. Database
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88},
    "103": {"name": "Ayan", "attendance": 100, "marks": 100}
}

# 3. LLM Agent Logic
def college_ai_agent(message, student_id, history):
    if history is None: history = []

    # Validation: Check if student exists
    if not student_id or student_id not in students:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "⚠️ Please enter a valid Student ID (e.g., 101, 102, 103)."})
        return "", history

    user_data = students[student_id]
    
    # --- STEP A: Clean History for Groq (Remove metadata) ---
    cleaned_history = [{"role": msg["role"], "content": msg["content"]} for msg in history]

    # --- STEP B: Build System Prompt (Giving the LLM its "Brain") ---
    system_prompt = {
        "role": "system",
        "content": (
            f"You are a helpful College Assistant. You are talking to {user_data['name']}. "
            f"Current Stats: Attendance is {user_data['attendance']}% and Marks are {user_data['marks']}/100. "
            "Answer questions based ONLY on this data. Be encouraging and brief."
        )
    }

    # --- STEP C: Call the Versatile Model ---
    api_messages = [system_prompt] + cleaned_history + [{"role": "user", "content": message}]

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=api_messages,
            temperature=0.6
        )
        bot_res = response.choices[0].message.content
    except Exception as e:
        bot_res = f"⚠️ Groq API Error: {str(e)}"

    # Update Gradio History
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_res})
    
    return "", history

# 4. Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎓 AI Student Support Portal")
    gr.Markdown("Enter your ID to discuss your attendance and academic performance.")
    
    with gr.Row():
        sid_input = gr.Textbox(label="Student ID", placeholder="101", scale=1)
    
    chatbot = gr.Chatbot(label="Chat with Assistant", type="messages")
    msg_input = gr.Textbox(label="Your Question", placeholder="How are my grades looking?")

    msg_input.submit(
        college_ai_agent, 
        inputs=[msg_input, sid_input, chatbot], 
        outputs=[msg_input, chatbot]
    )

demo.launch()

/tmp/ipykernel_1468/2413160997.py:68: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1468/2413160997.py:75: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat with Assistant", type="messages")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c829b1a7c99206e41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## SCENARIO: “Hospital Smart Assistant System”
### 🏥 Background Story
- A large hospital deploys an AI-powered patient assistant.
### 👉 Patients can ask:
- “What is my appointment schedule?”
- “What are my latest test results?”
### 👉 Instead of calling reception or logging into multiple portals,
### 👉 AI fetches it instantly, providing secure, real-time updates.

In [ ]:
import gradio as gr

# ================================
# STEP 1: Dummy Database (Patient Records)
# ================================
patients = {
    "P001": {"name": "Alice Smith", "appointment": "Oct 12, 10:30 AM with Dr. House", "results": "Vitamin D: Normal (32 ng/mL)"},
    "P002": {"name": "Bob Jones", "appointment": "Oct 15, 2:00 PM with Dr. Grey", "results": "Cholesterol: High (240 mg/dL)"}
}

# ================================
# STEP 2: Tool Functions
# ================================
def get_appointment(patient_id):
    if patient_id in patients:
        return f"📅 **Appointment:** {patients[patient_id]['appointment']}"
    return "❌ Patient ID not found."

def get_results(patient_id):
    if patient_id in patients:
        return f"🔬 **Latest Results:** {patients[patient_id]['results']}"
    return "❌ Patient ID not found."

# ================================
# STEP 3: Hospital Agent Logic
# ================================
def hospital_agent(message, patient_id, history):
    # Ensure history starts as a list
    if history is None:
        history = []

    # 1. ADD USER MESSAGE (Dictionary Format)
    history.append({"role": "user", "content": message})

    # 2. PROCESS LOGIC
    if not patient_id:
        response = "⚠️ Please enter your Patient ID to proceed."
    else:
        msg_clean = message.lower()
        if "appointment" in msg_clean or "schedule" in msg_clean:
            response = get_appointment(patient_id)
        elif "result" in msg_clean or "test" in msg_clean:
            response = get_results(patient_id)
        elif "hi" in msg_clean or "hello" in msg_clean:
            response = f"👋 Hello! I can help you with your appointment or test results."
        else:
            response = "🤖 I'm sorry, I can only check appointments and test results right now."

    # 3. ADD ASSISTANT RESPONSE (Dictionary Format)
    history.append({"role": "assistant", "content": response})

    # Return empty string to clear input, and the updated history
    return "", history

# ================================
# STEP 4: Gradio Interface
# ================================
with gr.Blocks(title="Hospital Assistant") as demo:
    gr.Markdown("# 🏥 Hospital Smart Assistant")
    gr.Markdown("Securely check your medical information instantly.")

    with gr.Row():
        patient_id = gr.Textbox(label="Patient ID (e.g., P001)", placeholder="Enter ID...")
    
    # We do NOT use type="messages" here to avoid your TypeError
    chatbot = gr.Chatbot(label="Hospital Concierge")
    msg = gr.Textbox(label="How can I help you today?", placeholder="Ask about appointments or results...")

    # Wire up the submission
    msg.submit(
        hospital_agent, 
        inputs=[msg, patient_id, chatbot], 
        outputs=[msg, chatbot]
    )

# ================================
# STEP 5: Launch
# ================================
demo.launch(share=True)

/tmp/ipykernel_1468/819218025.py:66: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Hospital Concierge")
/tmp/ipykernel_1468/819218025.py:66: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Hospital Concierge")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6e579588990fbd326.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# SCENARIO: “Banking Smart Assistant System”
### 🏦 Background Story
- A major bank launches an AI-powered customer assistant.
- 👉 Customers can ask:
- “What is my account balance?”
- “Show me my last 5 transactions.”
- “When is my loan EMI due?”
- 👉 Instead of logging into apps or waiting on customer service calls,
- 👉 AI fetches the information instantly, securely, and in real time.

### ⚙️ Core Idea:
Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
- 💡 Impact:
- Saves customers time.
- Reduces load on call centers.
- Provides personalized financial insights on demand.

In [12]:
import os
import gradio as gr
from openai import OpenAI

# 2. INITIALIZE CLIENT
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# 3. MOCK DATABASE
accounts = {
    "AC101": {"name": "Arjun", "balance": 50000, "emi": "Oct 10th"},
    "AC102": {"name": "Priya", "balance": 75000, "emi": "Oct 15th"}
}

# 4. AGENT LOGIC (Llama-3-Versatile)
def banking_ai(message, ac_id, history):
    if history is None: history = []
    
    if not ac_id or ac_id not in accounts:
        history.append({"role": "assistant", "content": "❌ Invalid Account ID."})
        return "", history

    user_data = accounts[ac_id]
    
    # 1. CLEAN HISTORY: Remove metadata that Groq doesn't support
    cleaned_history = []
    for msg in history:
        cleaned_history.append({
            "role": msg["role"], 
            "content": msg["content"]
        })

    # 2. PREPARE API MESSAGES
    api_messages = [
        {
            "role": "system", 
            "content": f"You are a Bank Assistant for {user_data['name']}. Balance: {user_data['balance']}, EMI: {user_data['emi']}."
        },
        *cleaned_history, # Use the cleaned version here
        {"role": "user", "content": message}
    ]

    try:
        # Using the current Llama 3.3 model
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile", 
            messages=api_messages
        )
        bot_res = response.choices[0].message.content
    except Exception as e:
        bot_res = f"⚠️ Groq Error: {str(e)}"

    # 3. UPDATE GRADIO (Gradio likes the full object, so append to the original history)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_res})
    
    return "", history

# 5. UI
with gr.Blocks() as demo:
    ac_input = gr.Textbox(label="Account ID (AC101)")
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(label="Message")
    msg.submit(banking_ai, [msg, ac_input, chatbot], [msg, chatbot])

demo.launch()

/tmp/ipykernel_1468/2621052063.py:64: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://399de29144a7bce006.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
import os
from openai import OpenAI


client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)


# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = "⚠️ Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    # Fallback (if LLM fails)
    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = "⚠️ LLM failed, and I couldn't understand."

    else:
        response = "🤖 I can help with attendance or marks."

    # Step 3: Save chat
    history.append((message, response))

    return "", history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 MCP Agent with Groq (Stable Version)")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch(share=True)

/tmp/ipykernel_579/2977724285.py:127: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_579/2977724285.py:127: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://026cb53a84a109d6d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	            Customer	Employee	Manager
# View own balance	        ✅	    ✅	        ✅
# View others' accounts	    ❌	    ✅	        ✅
# Approve loan	            ❌	    ❌	        ✅
# View all transactions	    ❌	    ✅	        ✅


# ======================================
# STEP 1: Install Libraries
# ======================================


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
import os
from openai import OpenAI


client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history.append((message, response))

    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=300)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch(share=True)

/tmp/ipykernel_1468/4231910838.py:189: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=300)
/tmp/ipykernel_1468/4231910838.py:189: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=300)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9458c0346f3624d328.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## SCENARIO: “University Smart Assistant with Role-Based Access”
### 🏫 Background Story
A university deploys an AI-powered academic assistant to help students, faculty, and administrators.
### 👉 Users can ask:
- “What is my attendance record?”
- “Show me my exam results.”
- “Update course schedules.”
- “Approve new course registrations.”
### 👉 But not everyone can do everything — access depends on roles.

In [15]:
import os
import gradio as gr
from openai import OpenAI

# ======================================
# STEP 1: Initialize Groq Client
# ======================================
# Ensure your secret is set as GROQ_API_KEY
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# ======================================
# STEP 2: University Database
# ======================================
university_db = {
    "S101": {"name": "Ayan", "attendance": 95, "marks": 88, "status": "Registered"},
    "S102": {"name": "Rahul", "attendance": 70, "marks": 65, "status": "Pending"},
}

# ======================================
# STEP 3: Tool Functions (APIs)
# ======================================
def get_academic_record(student_id):
    if student_id in university_db:
        data = university_db[student_id]
        return f"🎓 **Record for {data['name']}:** Attendance: {data['attendance']}%, Marks: {data['marks']}/100."
    return "❌ Student ID not found."

def update_schedule(course_name):
    return f"📅 **Schedule Updated:** The timetable for '{course_name}' has been refreshed."

def approve_registration(student_id):
    if student_id in university_db:
        university_db[student_id]["status"] = "Registered"
        return f"✅ **Approved:** Student {student_id} is now officially registered."
    return "❌ Student ID not found."

# ======================================
# STEP 4: RBAC Security Layer
# ======================================
def secure_access(role, user_id, target_id, action):
    # Admin -> Full Power
    if role == "admin":
        return True
    
    # Faculty -> Can view all records and update schedules, but not approve registration
    if role == "faculty":
        if action == "approve_registration":
            return False
        return True
    
    # Student -> Only own records, no updates, no approvals
    if role == "student":
        allowed_actions = ["get_academic_record"]
        if action in allowed_actions and user_id == target_id:
            return True
    
    return False

# ======================================
# STEP 5: LLM Intent Decision
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are a University Assistant. Categorize the user query into one of these actions:
        - get_academic_record (for attendance, marks, or results)
        - update_schedule (for changing times, classes, or timetables)
        - approve_registration (for enrolling or approving students)

        Return ONLY the action name.
        Query: {query}
        """
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

# ======================================
# STEP 6: Core Agent Logic
# ======================================
def university_agent(message, role, user_id, target_id, history):
    if history is None: history = []
    
    if not user_id:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "⚠️ Please enter your ID to continue."})
        return "", history

    if not target_id: target_id = user_id

    # 1. LLM Decides Intent
    action = decide_action(message)

    # 2. RBAC Check
    if not secure_access(role, user_id, target_id, action):
        response = f"🚫 **Access Denied:** As a {role}, you cannot perform: {action} on {target_id}."
    else:
        # 3. Tool Execution
        if "record" in action:
            response = get_academic_record(target_id)
        elif "schedule" in action:
            response = update_schedule("General Coursework")
        elif "approve" in action:
            response = approve_registration(target_id)
        else:
            response = "🤖 I can help with records, schedules, or approvals. How can I assist?"

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    return "", history

# ======================================
# STEP 7: Gradio UI
# ======================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏫 University Smart Assistant (RBAC + LLM)")
    
    with gr.Row():
        role_select = gr.Dropdown(["student", "faculty", "admin"], label="Select Role", value="student")
        u_id = gr.Textbox(label="Your ID (e.g., S101)", placeholder="Enter your ID")
        t_id = gr.Textbox(label="Target Student ID (Optional)", placeholder="Admin/Faculty use only")
    
    # Note: Using version-safe chatbot (dictionary format)
    chatbot = gr.Chatbot(label="Academic Support Chat", type="messages")
    msg = gr.Textbox(label="Message", placeholder="e.g., Show me my exam results")

    msg.submit(university_agent, [msg, role_select, u_id, t_id, chatbot], [msg, chatbot])

demo.launch()

/tmp/ipykernel_1468/2681497524.py:121: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1468/2681497524.py:130: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Academic Support Chat", type="messages")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2393e284038ba4e3b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## SCENARIO: “Retail Smart Assistant with Role-Based Access”
## 🏬 Background Story
A large retail chain introduces an AI-powered store assistant to streamline operations.
### 👉 Users can ask:
- “What is my purchase history?”
- “Check inventory for product X.”
- “Approve supplier orders.”
- “Manage employee schedules.”
### 👉 But not everyone can do everything — access depends on roles.

In [ ]:
import os
import gradio as gr
from openai import OpenAI

# ======================================
# STEP 1: Initialize Groq Client
# ======================================
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# ======================================
# STEP 2: Retail Database
# ======================================
inventory = {"Apples": 50, "Milk": 20, "Bread": 15}
purchases = {
    "C101": {"name": "Alice", "history": ["Milk", "Bread"]},
    "C102": {"name": "Bob", "history": ["Apples"]}
}

# ======================================
# STEP 3: Retail Tool Functions
# ======================================
def get_purchase_history(cid):
    if cid in purchases:
        return f"🛍️ **History for {cid}:** " + ", ".join(purchases[cid]["history"])
    return "❌ Customer ID not found."

def check_inventory(item):
    item_cap = item.capitalize()
    if item_cap in inventory:
        return f"📦 **Inventory:** {item_cap} has {inventory[item_cap]} units in stock."
    return "🔍 Item not found in current inventory."

def approve_order(order_id):
    return f"✅ **Supplier Order {order_id} Approved.** Logistics have been notified."

def manage_schedules():
    return "📅 **Staff Schedules:** Displaying shifts for the current week. All slots filled."

# ======================================
# STEP 4: RBAC Security Layer (Based on Image)
# ======================================
def secure_access(role, action):
    # Manager -> Unlimited access
    if role == "manager":
        return True
    
    # Staff -> Sales ops and inventory
    if role == "staff":
        allowed = ["view_own_history", "view_others_purchases", "check_inventory"]
        return action in allowed
    
    # Customer -> Only their own data
    if role == "customer":
        return action == "view_own_history"
    
    return False

# ======================================
# STEP 5: LLM Intent Decision
# ======================================
def decide_retail_action(query):
    try:
        prompt = f"""
        Retail AI. Map the query to one action:
        - view_own_history (purchases, bought, history)
        - view_others_purchases (what did someone else buy)
        - check_inventory (stock, units, count)
        - approve_supplier_orders (order, approve, buy from supplier)
        - manage_employee_schedules (shifts, roster, schedule)

        Return ONLY the action name.
        Query: {query}
        """
        res = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )
        return res.choices[0].message.content.strip().lower()
    except:
        return "fallback"

# ======================================
# STEP 6: Core Agent Logic
# ======================================
def retail_agent(message, role, user_id, history):
    if history is None: history = []
    
    # Clean history for API
    cleaned_history = [{"role": m["role"], "content": m["content"]} for m in history]

    # 1. LLM identifies the intent
    action = decide_retail_action(message)

    # 2. RBAC Check
    if not secure_access(role, action):
        response = f"🚫 **Access Denied:** As a {role}, you aren't authorized to perform: {action}."
    else:
        # 3. Execution logic
        if "history" in action:
            response = get_purchase_history(user_id)
        elif "inventory" in action:
            response = check_inventory("Milk") # Placeholder for entity extraction
        elif "order" in action:
            response = approve_order("ORD-99")
        elif "schedule" in action:
            response = manage_schedules()
        else:
            response = "🤖 I can help with history, inventory, orders, or schedules."

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    return "", history

# ======================================
# STEP 7: Gradio UI
# ======================================
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🏬 Retail Smart Assistant (RBAC + Llama 3.3)")
    
    with gr.Row():
        role_input = gr.Dropdown(["customer", "staff", "manager"], label="User Role", value="customer")
        uid_input = gr.Textbox(label="Customer/Staff ID", placeholder="C101")
    
    chatbot = gr.Chatbot(label="Retail Ops Channel", type="messages", height=400)
    msg = gr.Textbox(label="Assistant Terminal", placeholder="Ask: 'What is my history?' or 'Approve supplier orders'")

    msg.submit(retail_agent, [msg, role_input, uid_input, chatbot], [msg, chatbot])

demo.launch()

/tmp/ipykernel_1468/3708948563.py:120: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
/tmp/ipykernel_1468/3708948563.py:127: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Retail Ops Channel", type="messages", height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ce192525b2fed83b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
# SCENARIO: “Corporate Research Assistant System”
# 🏢 Background Story
# A multinational company deploys an AI-powered business intelligence assistant.
# 👉 Employees can ask:
# - “What’s the latest news about Tesla?”
# - “What’s Tesla’s current stock price?”
# - “Give me a company profile instantly.”
# 👉 Instead of manually searching news sites, finance portals, and HR databases,
# 👉 AI fetches all the data in parallel, analyzes it, and generates a professional report.

# ⚙️ How it works (mapped to your pipeline):
# - Parallel Data Collection → AI gathers news, stock prices, and company profiles simultaneously.
# - LLM Analysis → AI interprets the combined data, highlighting key insights.
# - Report Generation → AI produces a polished, executive-ready report.

# 💡 Impact:
# - Saves analysts hours of manual research.
# - Provides real-time, consolidated insights for decision-making.
# - Empowers managers with instant reports for board meetings or investor updates.



import os
import asyncio
import random
import nest_asyncio
import gradio as gr
from openai import OpenAI

# Apply nest_asyncio to allow nested event loops in notebook environments
nest_asyncio.apply()

# ======================================
# STEP 1: Initialize Groq Client
# ======================================
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# ======================================
# STEP 2: MOCK TOOLS (Simulating MCP Tools)
# ======================================
async def web_search(query):
    await asyncio.sleep(1)  # simulate network delay
    return f"📰 News about {query}: Market dominance is increasing amid new tech launches."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(150, 600)
    return f"📈 Stock price of {company}: ${price} (Up 2.4% today)"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"👥 {company} is a global leader with 10,000+ employees, HQ in New York."

# ======================================
# STEP 3: PARALLEL DATA COLLECTION
# ======================================
async def parallel_research(company):
    # Runs all three "tools" at the same time to save time
    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )
    news, stock, profile = results
    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }

# ======================================
# STEP 4: LLM CHAINING (Analysis & Report)
# ======================================
def analyse_and_report(data_text, company):
    # Step 1: Analysis
    analysis_res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"Analyze this business data and give key insights:\n{data_text}"}]
    )
    analysis = analysis_res.choices[0].message.content

    # Step 2: Professional Report Generation
    report_res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"Create a professional executive report for {company} based on this analysis:\n{analysis}"}]
    )
    return report_res.choices[0].message.content

# ======================================
# STEP 5: GRADIO COMPATIBLE AGENT
# ======================================
async def research_agent(message, history):
    # We treat the 'message' as the company name for this scenario
    company = message.strip()
    
    if not company:
        return history + [{"role": "assistant", "content": "⚠️ Please enter a company name to research."}]

    # 1. Parallel Research
    data = await parallel_research(company)
    
    combined_text = f"News: {data['news']}\nStock: {data['stock']}\nProfile: {data['profile']}"

    # 2. LLM Processing (Analysis -> Report)
    try:
        # Run synchronous OpenAI calls in a thread to keep UI responsive
        report = await asyncio.to_thread(analyse_and_report, combined_text, company)
    except Exception as e:
        report = f"⚠️ Research Error: {str(e)}"

    # 3. Update History (Modern Messages Format)
    history.append({"role": "user", "content": f"Generate report for {company}"})
    history.append({"role": "assistant", "content": report})
    
    return "", history

# ======================================
# STEP 6: GRADIO UI
# ======================================
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🏢 Corporate Research AI")
    gr.Markdown("Enter a company name to generate a real-time Business Intelligence report.")
    
    chatbot = gr.Chatbot(label="Executive Reports", type="messages", height=500)
    company_input = gr.Textbox(label="Company Name", placeholder="e.g. Tesla, NVIDIA, Black Rock")
    
    # Submit triggers the async pipeline
    company_input.submit(
        research_agent, 
        inputs=[company_input, chatbot], 
        outputs=[company_input, chatbot]
    )

demo.launch()

/tmp/ipykernel_1468/2463659044.py:124: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as demo:
/tmp/ipykernel_1468/2463659044.py:128: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Executive Reports", type="messages", height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b2786f0cf2bb95b44.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🏥 SCENARIO: “Healthcare Research Assistant System”
### 🩺 Background Story
A medical research institute deploys an AI-powered clinical intelligence assistant.
### 👉 Researchers and doctors can ask:
- • 	“What’s the latest research on diabetes treatments?”
- • 	“Summarize recent clinical trial results for cancer drugs.”
- • 	“Give me a profile of a pharmaceutical company instantly.”
### 👉 Instead of manually searching journals, trial databases, and company reports,
### 👉 AI fetches all the data in parallel, analyzes it, and generates a professional research summary.

### ⚙️ How it works (mapped to your pipeline):
- • 	Parallel Data Collection → AI gathers medical news, trial results, and company profiles simultaneously.
- • 	LLM Analysis → AI interprets the combined data, highlighting key medical insights.
- • 	Report Generation → AI produces a polished, researcher-ready report.

In [21]:
import os
import asyncio
import random
import nest_asyncio
import gradio as gr
from openai import OpenAI

# Required for running async code in Notebooks/Colab
nest_asyncio.apply()

# ======================================
# STEP 1: Initialize Groq Client
# ======================================
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# ======================================
# STEP 2: MOCK MEDICAL TOOLS (Asynchronous)
# ======================================
async def fetch_medical_journals(topic):
    await asyncio.sleep(1.5)  # Simulate deep search in PubMed/The Lancet
    return f"🔬 Recent Journals on {topic}: Breakthrough in SGLT2 inhibitors and cellular regeneration."

async def fetch_clinical_trials(drug_type):
    await asyncio.sleep(1.5)
    status = random.choice(["Phase II Success", "Phase III Recruiting", "Awaiting FDA Approval"])
    return f"🧪 Clinical Trial Update: {drug_type} shows {status} with 15% improved efficacy."

async def fetch_pharma_profile(company):
    await asyncio.sleep(1.5)
    return f"🏢 {company} Profile: R&D budget $2.4B, specialized in Immunotherapy and Oncology."

# ======================================
# STEP 3: PARALLEL DATA COLLECTION
# ======================================
async def parallel_medical_research(query):
    # This gathers data from three sources at the same time
    results = await asyncio.gather(
        fetch_medical_journals(query),
        fetch_clinical_trials(query),
        fetch_pharma_profile(query),
        return_exceptions=True
    )
    journals, trials, pharma = results
    return {
        "journals": journals if not isinstance(journals, Exception) else "Journal data unavailable",
        "trials": trials if not isinstance(trials, Exception) else "Trial data unavailable",
        "pharma": pharma if not isinstance(pharma, Exception) else "Pharma profile unavailable"
    }

# ======================================
# STEP 4: MEDICAL ANALYSIS & REPORTING
# ======================================
def generate_medical_report(combined_data, query):
    # Step 1: High-Level Clinical Analysis
    analysis_res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"Analyze this clinical data and provide key medical insights:\n{combined_data}"}]
    )
    analysis = analysis_res.choices[0].message.content

    # Step 2: Formal Research Summary
    report_res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"Generate a professional Medical Research Summary for the topic '{query}' based on this analysis:\n{analysis}"}]
    )
    return report_res.choices[0].message.content

# ======================================
# STEP 5: GRADIO AGENT LOGIC
# ======================================
async def healthcare_research_agent(message, history):
    if not message.strip():
        return history + [{"role": "assistant", "content": "⚠️ Please enter a medical topic or company name."}]

    # 1. Start Parallel Fetching
    data = await parallel_medical_research(message)
    
    combined_raw_text = f"JOURNALS: {data['journals']}\nTRIALS: {data['trials']}\nPHARMA: {data['pharma']}"

    # 2. Run LLM Chaining (Analysis -> Report)
    try:
        # Using to_thread to keep the UI responsive during LLM generation
        final_report = await asyncio.to_thread(generate_medical_report, combined_raw_text, message)
    except Exception as e:
        final_report = f"⚠️ Medical API Error: {str(e)}"

    # 3. Format for Gradio Chatbot
    history.append({"role": "user", "content": f"Researching: {message}"})
    history.append({"role": "assistant", "content": final_report})
    
    return "", history

# ======================================
# STEP 6: UI LAYOUT
# ======================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 Healthcare Intelligence Portal")
    gr.Markdown("Search for clinical trials, journal summaries, and pharma R&D data.")
    
    chatbot = gr.Chatbot(label="Clinical Insights", type="messages", height=550)
    medical_query = gr.Textbox(label="Research Topic / Company", placeholder="e.g. Immunotherapy, Pfizer, Diabetes Phase III")
    
    medical_query.submit(
        healthcare_research_agent, 
        inputs=[medical_query, chatbot], 
        outputs=[medical_query, chatbot]
    )

demo.launch()

/tmp/ipykernel_1468/2318887240.py:99: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1468/2318887240.py:103: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Clinical Insights", type="messages", height=550)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://758918dff879f5983e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# 🖥️ SCENARIO: “AI IT Helpdesk Assistant in a Large Company”
# 🏢 Background Story

# A large company (like Infosys or TCS) has thousands of employees.

# 👉 Employees face issues daily:

# VPN not working
# System hacked
# Email access denied
# Network outage

# 👉 Instead of manual IT support, the company builds an:

# 🤖 AI IT Helpdesk Assistant





# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
import os
import asyncio
import random
import nest_asyncio
import gradio as gr
from openai import OpenAI
import logging

# Required for running async code in Notebooks/Colab
nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)

# ======================================
# STEP 1: Initialize Groq Client
# ======================================
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)


# ======================================
# STEP 3: Simulated MCP Tools (Mock APIs)
# ======================================
async def page_security_team(payload):
    return "🚨 Security team paged"

async def create_jira_ticket(payload):
    return f"🎫 Jira ticket created: {payload}"

async def check_network_status(payload):
    return random.choice(["Network degraded", "All systems normal"])

async def alert_noc_team(payload):
    return "📡 NOC team alerted"

async def get_ad_user(payload):
    return "user_123"

async def reset_permissions(payload):
    return "🔐 Permissions reset successfully"

async def query_postgres(payload):
    if random.random() < 0.7:
        raise Exception("Database timeout")
    return "📦 Data from Postgres"

async def query_sqlite_cache(payload):
    return "🗂 Data from SQLite cache (fallback)"


# ======================================
# STEP 4: LLM Classifier via Groq
# ======================================
def classify_issue(issue):
    prompt = f"""
    Classify the following IT issue into ONE category only:
    network, hardware, software, security, access

    Issue: {issue}

    Return only the category name.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()


# ======================================
# STEP 5: Conditional Routing Logic
# ======================================
async def smart_it_triage(issue, severity):
    issue_type = classify_issue(issue)

    print(f"🧠 Classified as: {issue_type}")

    if issue_type == "security":
        await page_security_team({"issue": issue})
        return await create_jira_ticket({"project": "SEC", "priority": "Blocker"})

    elif issue_type == "network" and severity == "high":
        status = await check_network_status({})
        if "degraded" in status.lower():
            return await alert_noc_team({"issue": issue})
        else:
            return await create_jira_ticket({"project": "NET", "priority": "High"})

    elif issue_type == "access":
        user = await get_ad_user({"query": issue})
        return await reset_permissions({"user": user})

    else:
        return await create_jira_ticket({"project": "IT", "priority": "Medium"})


# ======================================
# STEP 6: Retry with Exponential Backoff
# ======================================
async def call_tool_with_retry(
    primary_tool,
    fallback_tool=None,
    max_retries=3
):
    delay = 1

    for attempt in range(max_retries):
        try:
            return await primary_tool({})
        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed: {e}")

            if attempt == max_retries - 1:
                if fallback_tool:
                    logging.info("Switching to fallback tool")
                    return await fallback_tool({})
                raise

            await asyncio.sleep(delay)
            delay *= 2


# ======================================
# STEP 7: Run Demo
# ======================================
async def run_demo():
    print("=== Smart IT Triage Demo ===")

    result = await smart_it_triage(
        issue="VPN not working for employee",
        severity="high"
    )

    print("🔍 Routing Result:", result)

    print("\n=== Retry Pattern Demo ===")

    data = await call_tool_with_retry(
        query_postgres,
        fallback_tool=query_sqlite_cache
    )

    print("📊 Data Result:", data)


# ======================================
# STEP 8: Execute (Colab-safe)
# ======================================
await run_demo()

=== Smart IT Triage Demo ===
🧠 Classified as: security
🔍 Routing Result: 🎫 Jira ticket created: {'project': 'SEC', 'priority': 'Blocker'}

=== Retry Pattern Demo ===
📊 Data Result: 📦 Data from Postgres
